# Prompt-only pricing agent: comparing prompting strategies

Stage 1 of the retail pricing agent: recommend a selling price for one product using **only the model's own reasoning** — no tools, no data lookup, and no code that checks the numbers.

**Inputs:** cost price, current price, target margin (%), competitor price and price elasticity (Low / Medium / High).

This notebook runs four prompting strategies on the same reference product and compares how stable, checkable and correct their recommendations are:

| Strategy | Idea |
|---|---|
| Basic | one instruction with the inputs filled in |
| Chain-of-thought | an explicit margin formula, four analysis steps and a fixed output format |
| Few-shot | two worked examples, no explicit rules |
| Agent | a system persona with pricing rules, plus a step-by-step prompt |

The prompts and the `PricingAgent` class live in [`pricing_agent.py`](pricing_agent.py). The notebook imports them, so the notebook and the command-line tool always run the same prompts.

## Setup

Install the dependencies from the repository root (`pip install -r requirements.txt`) and put your Groq API key in a `.env` file at the repository root (copy `.env.example`). `pricing_agent` loads the key when it is imported.

In [1]:
from pricing_agent import (
    AGENT_SYSTEM_PROMPT, BASIC_PROMPT, COT_PROMPT, EXAMPLE_PRODUCT, FEW_SHOT_PROMPT, MODEL,
    PricingAgent, make_llm,
)

llm = make_llm()  # temperature 0.2, max_tokens 4096
print("Model:", MODEL)
print("Reference product:", EXAMPLE_PRODUCT)

Model: openai/gpt-oss-120b
Reference product: {'cost_price': 400, 'current_price': 599, 'target_margin': 25, 'competitor_price': 579, 'price_elasticity': 'Medium'}


## Ground truth for the reference product

The agent must not use tools, but *we* can: the arithmetic every answer should agree with, computed in Python, so each recommendation below can be checked. Margin is gross margin on selling price, `(price - cost) / price`.

In [2]:
def margin(price, cost=EXAMPLE_PRODUCT["cost_price"]):
    """Gross margin on selling price."""
    return (price - cost) / price

cost = EXAMPLE_PRODUCT["cost_price"]
current = EXAMPLE_PRODUCT["current_price"]
competitor = EXAMPLE_PRODUCT["competitor_price"]
target = EXAMPLE_PRODUCT["target_margin"] / 100

floor = cost / (1 - target)
print(f"Margin floor (lowest price that keeps a {target:.0%} margin): ${floor:.2f}")
print(f"Current margin at ${current}: {margin(current):.1%}")
print(f"Margin if we match the competitor at ${competitor}: {margin(competitor):.1%}")
print(f"Current price vs competitor: ${current - competitor} above ({(current - competitor) / competitor:.1%})")

Margin floor (lowest price that keeps a 25% margin): $533.33
Current margin at $599: 33.2%
Margin if we match the competitor at $579: 30.9%
Current price vs competitor: $20 above (3.5%)


## 1. Basic prompt

The task, the five inputs and a request for a price with a brief explanation. No formula, no steps, no output format.

In [3]:
print(BASIC_PROMPT.template)

You are a retail pricing agent. Recommend an optimal selling price balancing margin and competition.

Inputs:
- Cost Price: ${cost_price}
- Current Price: ${current_price}
- Target Margin: {target_margin}%
- Competitor Price: ${competitor_price}
- Price Elasticity: {price_elasticity}

Give the recommended selling price and a brief explanation of your reasoning.


In [4]:
basic_answer = (BASIC_PROMPT | llm).invoke(EXAMPLE_PRODUCT).content
print(basic_answer)

**Recommended selling price: $575**

**Why this works**

| Item | Value |
|------|-------|
| **Cost price** | $400 |
| **Target margin** | 25 % → minimum price to hit the target = $400 ÷ (1‑0.25) = **$533.33** |
| **Competitor price** | $579 |
| **Current price** | $599 |
| **Price elasticity** | Medium (price changes will affect demand, but not dramatically) |

1. **Margin safety‑net** – $575 is well above the $533.33 break‑even for a 25 % margin, delivering a **30 % gross margin** ( ($575‑$400)/$575 ≈ 30 %). This comfortably exceeds the target while leaving room for promotional discounts if needed.

2. **Competitive positioning** – The competitor is at $579. Pricing at $575 puts you **$4 lower**, giving a clear price advantage without triggering a race‑to‑the‑bottom. It’s close enough to the market reference that customers still perceive the product as comparable in quality.

3. **Elasticity consideration** – With medium elasticity, a modest $4‑$5 price cut can boost volume modestly,

## 2. Chain-of-thought prompt

Adds a role, defines margin explicitly (on selling price, so the minimum price is `cost / (1 - margin)`), lists four analysis steps in a fixed order and fixes the last three lines of the answer.

In [5]:
print(COT_PROMPT.template)

ROLE: You are an experienced retail pricing analyst.
TASK: Recommend ONE optimal selling price that balances profit margin and competitiveness.

PRODUCT DATA:
- Cost Price: ${cost_price}
- Current Selling Price: ${current_price}
- Target Margin: {target_margin}% (gross margin on selling price = (price - cost) / price)
- Competitor Price: ${competitor_price}
- Price Elasticity of Demand: {price_elasticity} (Low = customers barely react to price changes, High = customers are very price-sensitive)

ANALYSIS STEPS (think step by step and show your work):
1. Calculate the minimum price based on target margin: minimum price = cost / (1 - target margin). Also compute the current margin.
2. Analyze competitive positioning: compare the current price and the minimum price with the competitor price (gap in $ and %).
3. Consider price elasticity impact: explain how sensitive demand is and whether we should price above, at, or below the competitor.
4. Recommend the optimal price: pick a single pric

In [6]:
cot_answer = (COT_PROMPT | llm).invoke(EXAMPLE_PRODUCT).content
print(cot_answer)

**1. Minimum Price & Current Margin**  
- Minimum price to meet 25 % target margin: $400 ÷ (1 – 0.25) = **$533.33**  
- Current margin at $599: (599 – 400) ÷ 599 ≈ **33.2 %**

**2. Competitive Positioning**  
- Current price vs. competitor: $599 – $579 = +$20 (≈ +3.5 %)  
- Minimum price vs. competitor: $533.33 – $579 = –$45.67 (≈ –7.9 %)

**3. Elasticity Impact**  
- Medium elasticity means demand will respond moderately to price changes.  
- A modest price reduction can boost sales without eroding margin dramatically, so pricing just below the competitor is advisable.

**4. Recommended Price**  
- Set price at **$574**, which is $5 below the competitor, still above the $533.33 floor, and yields a solid margin.

Recommended Price: $574  
Expected Margin: 30.4%  
Summary: A modest $5 discount below the competitor keeps a healthy 30 % margin while improving competitiveness given medium price sensitivity.


## 3. Few-shot prompt

No rules at all: two worked pricing decisions, then the new case. The model has to infer the pricing policy from the examples.

In [7]:
print(FEW_SHOT_PROMPT.template)

You are a pricing expert. Here are some examples of good pricing decisions:

Example 1:
Cost: $200, Current: $400, Target Margin: 30%, Competitor: $380, Elasticity: Medium
Recommendation: $375 (maintains margin above 30%, competitive with market, good for medium elasticity)

Example 2:
Cost: $100, Current: $180, Target Margin: 25%, Competitor: $200, Elasticity: Low
Recommendation: $190 (exceeds margin target, leverages low elasticity for higher profit)

Now analyze this case:
Cost: ${cost_price}, Current: ${current_price}, Target Margin: {target_margin}%, Competitor: ${competitor_price}, Elasticity: {price_elasticity}

Recommendation:


In [8]:
few_shot_answer = (FEW_SHOT_PROMPT | llm).invoke(EXAMPLE_PRODUCT).content
print(few_shot_answer)

**Recommendation:** **$575**

**Why $575 makes sense**

| Factor | Calculation / Insight | Decision Impact |
|--------|-----------------------|-----------------|
| **Current margin** | \((599‑400) ÷ 599 ≈ 33.2\%\) | Well above the 25 % target, so we have room to cut price without jeopardising profitability. |
| **Minimum price to hit target margin** | \(400 ÷ (1‑0.25) = 400 ÷ 0.75 = **$533.33**\) | Any price **≥ $534** still meets the 25 % margin goal. |
| **Competitor price** | $579 | To stay competitive we need to be at or below this level. |
| **Elasticity (Medium)** | Demand reacts moderately to price changes. A modest price drop can boost volume without triggering a price war. | A small discount (≈ $4–$9 below the competitor) is enough to win price‑sensitive buyers while preserving healthy margins. |
| **Profitability at $575** | Margin = \((575‑400) ÷ 575 ≈ 30.4\%\) | Still comfortably above the 25 % target, giving a solid cushion for cost fluctuations or promotional spend. |

##

## 4. Agent with a system persona

`PricingAgent` puts the persona and the pricing rules in a **system message**, which applies to every request, and sends the product in a step-by-step user prompt.

In [9]:
print(AGENT_SYSTEM_PROMPT)

You are PriceWise, a senior retail pricing strategist.
You recommend optimal selling prices using ONLY the numbers the user gives you - no tools, no market data lookup.

Your principles:
- Never recommend a price below the minimum price needed to hit the target margin.
- Gross margin is measured on selling price: margin = (price - cost) / price, so minimum price = cost / (1 - target margin).
- Use the competitor price as the market anchor.
- Low elasticity: customers are not price-sensitive, so you may price at or above the competitor.
- Medium elasticity: stay close to the competitor price.
- High elasticity: customers are very price-sensitive, so price at or slightly below the competitor if the margin allows.
- Always show your calculations and reasoning step by step, clearly and concisely.
- Recommend exactly one price, not a range.


In [10]:
agent = PricingAgent(llm)
agent_answer = agent.get_price_recommendation(**EXAMPLE_PRODUCT)
print(agent_answer)

**1. Margin Floor**  
- Minimum price to hit 25 % margin:  
  \[
  \text{Floor} = \frac{\text{Cost}}{1-\text{Target Margin}} = \frac{400}{1-0.25}= \frac{400}{0.75}= \$533.33
  \]  
- Current margin at $599:  
  \[
  \frac{599-400}{599}= \frac{199}{599}=0.332\;(33.2\%)
  \]

**2. Competitive Position**  
- Competitor price = $579 (margin ≈ 30.9 %).  
- Current price $599 is **$20 above** the competitor and well above the margin floor ($533.33).  

**3. Elasticity Impact (Medium)**  
- With medium elasticity we should stay close to the market anchor (the competitor price).  
- We can price at or just above $579 while still comfortably exceeding the 25 % margin requirement.

**4. Recommendation**  
- Choose a price a few dollars above the competitor to capture a modest premium but remain “close.”  
- **Proposed price:** **$585**  

  - Margin at $585: \((585-400)/585 = 185/585 = 0.316\) → **31.6 %**  
  - Gap vs competitor: **$585 – $579 = $6** (slightly above)

---

**Recommended Price:*

### The agent on harder cases

Three cases that probe the rules: high elasticity, low elasticity with the competitor priced above us, and a competitor priced **below our margin floor**, where matching the competitor would break the margin target.

In [11]:
SCENARIOS = {
    "High elasticity": dict(cost_price=100, current_price=200, target_margin=30, competitor_price=180, price_elasticity="High"),
    "Low elasticity, competitor above us": dict(cost_price=50, current_price=100, target_margin=40, competitor_price=120, price_elasticity="Low"),
    # floor = 300 / (1 - 0.35) = $461.54, but the competitor sells at $420
    "Competitor below our margin floor": dict(cost_price=300, current_price=450, target_margin=35, competitor_price=420, price_elasticity="Medium"),
}

scenario_answers = {}
for name, product in SCENARIOS.items():
    floor = product["cost_price"] / (1 - product["target_margin"] / 100)
    print("=" * 70)
    print(f"{name}  (margin floor ${floor:.2f}, competitor ${product['competitor_price']})")
    print("=" * 70)
    scenario_answers[name] = agent.get_price_recommendation(**product)
    print(scenario_answers[name], end="\n\n")

High elasticity  (margin floor $142.86, competitor $180)


**1. Margin Floor**  
- Minimum price to hit a 30 % margin:  
\[
\text{Margin Floor} = \frac{\text{Cost}}{1-\text{Target Margin}} = \frac{100}{1-0.30}= \frac{100}{0.70}= \$142.86
\]  
- Current margin at the $200 price:  
\[
\text{Current Margin} = \frac{200-100}{200}=0.50 = 50\%
\]

**2. Competitive Position**  
- Current price ($200) is **$20 above** the competitor’s $180.  
- Margin floor ($142.86) is **$37.14 below** the competitor’s price, meaning we have room to lower the price while still meeting the 30 % margin target.

**3. Elasticity Impact**  
- Elasticity is **High** → customers are price‑sensitive.  
- Strategy: price at or just below the competitor, but never below the margin floor.  
- The highest price we can set that is still below the competitor is just under $180.  

**4. Recommendation**  
- Choose a price **slightly below** the competitor: **$179.00**.  
- Expected margin:  
\[
\frac{179-100}{179}= \frac{79}{179}\approx 0.4413 = 44.1\%
\]  
- Gap vs competitor: **$

**1. Margin Floor**  
- Minimum price to hit a 40 % margin:  
  \[
  \text{Floor} = \frac{\text{Cost}}{1-\text{Target Margin}} = \frac{50}{1-0.40}= \frac{50}{0.60}= \$83.33
  \]  
- Current margin at the $100 price:  
  \[
  \text{Current Margin}= \frac{100-50}{100}=0.50 = 50\%
  \]

**2. Competitive Position**  
- Competitor price: **$120**  
- Current price ($100) is **$20 below** the competitor and **$16.67 above** the margin floor.  
- The margin floor ($83.33) is also well below the competitor price, giving ample room to move upward.

**3. Elasticity Impact**  
- Elasticity is **Low** → customers are not price‑sensitive.  
- We can price **at or above** the competitor without risking significant sales loss, provided we stay above the margin floor (which we will).

**4. Recommendation**  
- Set the price **slightly above** the competitor to capture extra margin while still being acceptable to low‑elasticity buyers.  

**Recommended Price:** **$125**  
**Expected Margin:** **60 %** 

**1. Margin Floor**  
- Minimum price to hit a 35 % margin:  
\[
\text{Floor} = \frac{\text{Cost}}{1-\text{Target Margin}} = \frac{300}{1-0.35}= \frac{300}{0.65}= \$461.54
\]  
- Current margin at the $450 price:  
\[
\text{Current Margin}= \frac{450-300}{450}= \frac{150}{450}=0.3333 = 33.33\%
\]

**2. Competitive Position**  
- Current price ($450) is **above** the competitor ($420) but **below** the margin floor ($461.54).  
- The margin floor is higher than both the competitor and the current price, meaning we cannot meet the 35 % target without raising the price above the competitor’s level.

**3. Elasticity Impact (Medium)**  
- With medium elasticity we aim to stay close to the competitor’s price, but we must respect the margin floor.  
- The closest we can get to the competitor while still achieving the target margin is just at the margin floor (or a minimal rounding up).

**4. Recommendation**  
- Set the price at the margin‑floor level, rounded to a practical retail figure: **

### A quick check without the system message

`quick_price_check` sends a three-input prompt **without** the system message — the same model, minus the persona and rules.

In [12]:
print(agent.quick_price_check(cost_price=250, target_margin=20, competitor_price=350))

**Recommended price:** **≈ $315**

**Why:**

| Item | Value |
|------|-------|
| Cost | $250 |
| Target margin | 20 % |
| Minimum price to hit margin | $250 ÷ (1 – 0.20) = **$312.50** |
| Competitor’s price | $350 |

- **Margin goal:** Pricing at $312.50 meets the 20 % margin exactly.  
- **Market positioning:** Setting the price a few dollars above the margin floor (e.g., $315) keeps you **well below the competitor’s $350**, giving a clear price advantage while still preserving the desired margin.  
- **Round‑off:** $315 is a clean, market‑friendly figure that still yields a margin of about **21 %** ( $315 – $250 = $65 profit → $65 ÷ $315 ≈ 20.6 % ).

**Bottom line:** Price at **$315** to achieve your margin target and stay competitively priced.


## 5. Comparison: the same input, run a second time

Output varies between runs (temperature 0.2), so each strategy is run once more on the reference product. The table puts the recommended price from both runs side by side, with its margin computed in Python. Prices are pulled from the free text with the small pattern match below, and the line each price came from is printed under the table so every value can be checked. The last column shows whether the answer contains the exact `Recommended Price: $<number>` line that the chain-of-thought and agent prompts ask for — the kind of line code would rely on.

In [13]:
import re

def find_recommendation(answer):
    """Return (price, source line) for the recommended price in a free-text answer."""
    lines = answer.replace("*", "").splitlines()  # ignore markdown bold
    for line in lines:
        found = re.search(r"Recommended Price:\s*\$\s?([\d,]+(?:\.\d+)?)", line, re.IGNORECASE)
        if found:
            return float(found.group(1).replace(",", "")), line.strip()
    for line in lines:  # otherwise: the first $ amount on a line about the recommendation
        if "recommend" in line.lower():
            found = re.search(r"\$\s?([\d,]+(?:\.\d+)?)", line)
            if found:
                return float(found.group(1).replace(",", "")), line.strip()
    return None, None

def exact_format_line(answer):
    return bool(re.search(r"^Recommended Price: \$[\d,.]+\s*$", answer, re.MULTILINE))

first_run = {"Basic": basic_answer, "Chain-of-thought": cot_answer, "Few-shot": few_shot_answer, "Agent": agent_answer}
second_run = {
    "Basic": (BASIC_PROMPT | llm).invoke(EXAMPLE_PRODUCT).content,
    "Chain-of-thought": (COT_PROMPT | llm).invoke(EXAMPLE_PRODUCT).content,
    "Few-shot": (FEW_SHOT_PROMPT | llm).invoke(EXAMPLE_PRODUCT).content,
    "Agent": agent.get_price_recommendation(**EXAMPLE_PRODUCT),
}
asks_for_format = {"Chain-of-thought", "Agent"}

def show(price):
    return f"${price:,.2f} ({margin(price):.1%})" if price else "not found"

print(f"{'Strategy':<18}{'Run 1 (margin)':<22}{'Run 2 (margin)':<22}Exact format line (run 1 / run 2)")
for name in first_run:
    p1, p2 = find_recommendation(first_run[name])[0], find_recommendation(second_run[name])[0]
    fmt = (f"{'yes' if exact_format_line(first_run[name]) else 'no'} / {'yes' if exact_format_line(second_run[name]) else 'no'}"
           if name in asks_for_format else "n/a (not requested)")
    print(f"{name:<18}{show(p1):<22}{show(p2):<22}{fmt}")

print("\nWhere each price was found:")
for name in first_run:
    for run, answers in (("run 1", first_run), ("run 2", second_run)):
        print(f"  {name} {run}: {find_recommendation(answers[name])[1]}")

Strategy          Run 1 (margin)        Run 2 (margin)        Exact format line (run 1 / run 2)
Basic             $575.00 (30.4%)       $549.00 (27.1%)       n/a (not requested)
Chain-of-thought  $574.00 (30.3%)       $579.00 (30.9%)       yes / yes
Few-shot          $575.00 (30.4%)       $575.00 (30.4%)       n/a (not requested)
Agent             $585.00 (31.6%)       $585.00 (31.6%)       no / no

Where each price was found:
  Basic run 1: Recommended selling price: $575
  Basic run 2: Recommended selling price: $549
  Chain-of-thought run 1: Recommended Price: $574
  Chain-of-thought run 2: Recommended Price: $579
  Few-shot run 1: Recommendation: $575
  Few-shot run 2: Recommendation: $575
  Agent run 1: Recommended Price: $585
  Agent run 2: - Recommended price: $585


## Observations

All numbers below come from the run saved in this notebook, and every margin was checked against the ground-truth cell. Output varies between runs, so re-running the notebook will give somewhat different prices.

| Strategy | Run 1 | Run 2 | What stood out |
|---|---|---|---|
| **Basic** | $575 | $549 | The least stable: the same input gave prices $26 apart. With no steps and no output format, the price can appear anywhere in the answer. |
| **Chain-of-thought** | $574 | $579 | Every step is shown, so every number can be checked, and both runs end with the exact `Recommended Price:` line. It is still not free of arithmetic slips: run 1 reports a 30.4% margin for $574, which is really 30.3%. |
| **Few-shot** | $575 | $575 | Stable, because it copies the examples' pattern — a few dollars under the competitor, like $375 against $380 in Example 1 — without being given any rules. Its own what-if table claims a *lower* price of $560 gives "less competitive advantage vs. $579", the opposite of the truth. |
| **Agent** (system persona) | $585 | $585 | Stable, and its arithmetic was correct throughout, including the three harder cases. It is the only strategy that priced *above* the competitor: its rule "medium elasticity: stay close to the competitor price" does not say which side, and it chose a $6 premium. |

**Key takeaways**

1. **Structure buys stability.** The two strategies that constrain the model most — worked examples, or a system persona with explicit rules — gave the same price on both runs. The bare prompt moved by $26.
2. **Define terms explicitly, and word rules precisely.** Every answer here measured margin on selling price (floor $533.33), but only the chain-of-thought and agent prompts guarantee that by stating the formula; "margin" could also be read as markup on cost, which gives a $500 floor. Rules are only as precise as their wording: "stay close to the competitor" let the agent price above it while every other strategy went below.
3. **The system message carries the agent's discipline.** On the hardest case — a competitor at $420, below our $461.54 margin floor — the agent held the price at $462 (35.1% margin) rather than matching the competitor. Without the system message, `quick_price_check` anchored on cost instead of the market: it priced just above the $312.50 floor, $35 under the competitor, and hedged the number ("≈ $315", "e.g., $315").
4. **Free text is fragile to consume.** Asked to end with `Recommended Price: $<number>`, the agent wrapped the line in markdown bold (`**Recommended Price:** $585`) on both runs. Code that reads the price has to pattern-match prose, as the comparison cell does.
5. **Prompt-only means unchecked arithmetic.** The model does every calculation itself and nothing verifies it. In this run the slips were small (a margin off by 0.1 points) or sat in side commentary (the few-shot what-if table), but the agent has no way to catch them.

## Limitations and next steps

- **No verification.** The arithmetic and the business rules (never below the margin floor, one price) are enforced only by the prompt. A typed, structured recommendation would let code validate the price before it is used.
- **No tools.** Exact calculations such as the margin floor and the margin at a candidate price are a natural fit for tool calls instead of the model's mental arithmetic.
- **No data.** The competitor price and the elasticity are single numbers typed in by the user. A grounded agent would retrieve product, sales and competitor data instead.
- **One reference case.** The comparison above uses one product and two runs. Judging a strategy properly needs an evaluation set of many products with known acceptable price bands.

These are the directions for the next stages of this project.